[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Perform OCR with GLM-OCR (Vision AI Checkup Benchmark)

---

[![arXiv](https://img.shields.io/badge/arXiv-2603.10910-b31b1b.svg)](https://arxiv.org/abs/2603.10910)

[GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) is a 0.9B-parameter
vision-language model built for optical character recognition. It scores
94.62 on OmniDocBench V1.5, placing it first on that benchmark. The
architecture pairs a CogViT visual encoder with a GLM-0.5B language
decoder through a cross-modal connector that downsamples visual tokens.

The model handles documents at resolutions up to 8K (7680x4320) in 8+
languages. Three built-in modes cover text recognition, LaTeX formula
recognition, and table recognition. Custom prompts can extract
structured data like JSON from documents.

In this notebook we evaluate GLM-OCR on **OCR** and **Document
Understanding** cases drawn from the
[Vision AI Checkup](https://visioncheckup.com) benchmark by Roboflow.
Each case uses the exact same image and prompt from the benchmark,
letting you compare GLM-OCR results against
[other models on the leaderboard](https://visioncheckup.com).

## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi`
command to do that. In case of any problems navigate to `Edit` →
`Notebook settings` → `Hardware accelerator`, set it to `L4 GPU`,
and then click `Save`.

In [ ]:
!nvidia-smi

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    gpu_name = torch.cuda.get_device_name()
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float16
    gpu_name = "Apple Silicon (MPS)"
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32
    gpu_name = "CPU"

print(f"Device : {DEVICE} ({gpu_name})")
print(f"Dtype  : {DTYPE}")

### Configure your API keys

To run this notebook you need a HuggingFace Token (to download the
model). Follow these steps:

- Open your [`HuggingFace Settings`](https://huggingface.co/settings) page. Click `Access Tokens` then `New Token` to generate new token.
- In Colab, go to the left pane and click on `Secrets` (🔑).
    - Store HuggingFace Access Token under the name `HF_TOKEN`.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## Install dependencies

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git

## Load GLM-OCR model

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "zai-org/GLM-OCR"

processor = AutoProcessor.from_pretrained(MODEL_ID)

if DEVICE.type == "cuda":
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        device_map="auto",
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
    ).to(DEVICE)

## Define helper functions

In [ ]:
import io
import textwrap
from urllib.request import urlopen

import matplotlib.pyplot as plt
from PIL import Image


def download_image(url: str) -> Image.Image:
    """Download an image from a URL and return as a PIL Image."""
    with urlopen(url) as resp:
        return Image.open(io.BytesIO(resp.read())).convert("RGB")


def run_ocr(image: Image.Image, prompt: str = "Text Recognition:") -> str:
    """Run GLM-OCR on a single PIL image and return the decoded text."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    inputs.pop("token_type_ids", None)
    generated_ids = model.generate(**inputs, max_new_tokens=8192)
    return processor.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()


def display_result(
    image: Image.Image,
    expected: str,
    prediction: str,
    wrap_width: int = 60,
):
    """Display image side-by-side with expected and predicted answers."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(image)
    ax.axis("off")

    expected_fmt = textwrap.fill(str(expected), width=wrap_width)
    predicted_fmt = textwrap.fill(prediction, width=wrap_width)

    title = (
        f"Expected : {expected_fmt}\n"
        f"Predicted: {predicted_fmt}"
    )
    ax.set_title(
        title.replace("$", "\\$"),
        fontsize=9,
        loc="left",
        pad=8,
        family="monospace",
    )
    plt.tight_layout()
    plt.show()


results = []

## OCR

The [Vision AI Checkup](https://visioncheckup.com) benchmark includes
nine OCR challenges: reading serial numbers from tires and containers,
extracting text from screenshots, reading shipping manifests, barcodes,
part numbers, supermarket shelf labels, and shipping container weight
plates.

### 1. Read a serial number

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/tire.png")

prediction = run_ocr(image, prompt="What is the serial number on the tire? Answer only the serial number.")

display_result(
    image=image,
    expected="3702692432",
    prediction=prediction,
)

results.append({"name": "Read a serial number", "expected": "3702692432", "prediction": prediction})

### 2. Read a screenshot of prose

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/ocr.png")

prediction = run_ocr(image, prompt="Read the text in the image. Return only unformatted text.")

display_result(
    image=image,
    expected="I was thinking earlier today that I have gone through, to use the lingo, eras of listening to each of Swift\u2019s Eras. Meta indeed. I started listening to Ms. Swift\u2019s music after hearing the Midnights album. A few weeks after hearing the album for the first time, I found myself playing various songs on repeat. I listened to the album in order multiple times.",
    prediction=prediction,
)

results.append({"name": "Read a screenshot of prose", "expected": "I was thinking earlier today that I have gone through, to use the lingo, eras of listening to each of Swift\u2019s Eras. Meta indeed. I started listening to Ms. Swift\u2019s music after hearing the Midnights album. A few weeks after hearing the album for the first time, I found myself playing various songs on repeat. I listened to the album in order multiple times.", "prediction": prediction})

### 3. Shipping manifest reading

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/shipping_manifest.png")

prediction = run_ocr(image, prompt="What is the dock number? Return like AAA12.")

display_result(
    image=image,
    expected="D33",
    prediction=prediction,
)

results.append({"name": "Shipping manifest reading", "expected": "D33", "prediction": prediction})

### 4. Shipping container OCR

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/container_id_ocr.png")

prediction = run_ocr(image, prompt="What is the horizontal ID on the container? Return only the ID.")

display_result(
    image=image,
    expected="JBHU282862",
    prediction=prediction,
)

results.append({"name": "Shipping container OCR", "expected": "JBHU282862", "prediction": prediction})

### 5. Supermarket shelf label reading (one item)

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/price.png")

prediction = run_ocr(image, prompt="How much are the wipes? Currency is GBP. Return in form 1.00, or 0.00.")

display_result(
    image=image,
    expected="0.58",
    prediction=prediction,
)

results.append({"name": "Supermarket shelf label reading (one item)", "expected": "0.58", "prediction": prediction})

### 6. Supermarket shelf label reading (two items)

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/price_with_two_items.png")

prediction = run_ocr(image, prompt="How much are the wipes? Currency is GBP. Return in form 1.00, or 0.00.")

display_result(
    image=image,
    expected="0.58",
    prediction=prediction,
)

results.append({"name": "Supermarket shelf label reading (two items)", "expected": "0.58", "prediction": prediction})

### 7. Read barcode ID on circuit board

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/barcode_reading.png")

prediction = run_ocr(image, prompt="What is the ID on the barcode? Return only the ID text.")

display_result(
    image=image,
    expected="T074802630B2",
    prediction=prediction,
)

results.append({"name": "Read barcode ID on circuit board", "expected": "T074802630B2", "prediction": prediction})

### 8. Part number reading

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/part_number.png")

prediction = run_ocr(image, prompt="Return the part number.")

display_result(
    image=image,
    expected="10629101",
    prediction=prediction,
)

results.append({"name": "Part number reading", "expected": "10629101", "prediction": prediction})

### 9. Maximum gross weight label on shipping container

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/container_information_retrieval.png")

prediction = run_ocr(image, prompt="What is the max gross weight? Return only number (value in KG), without separators/decimals, like \"1234\"")

display_result(
    image=image,
    expected="32500",
    prediction=prediction,
)

results.append({"name": "Maximum gross weight label on shipping container", "expected": "32500", "prediction": prediction})

## Document Understanding

The benchmark also tests document understanding across nine cases:
reading floor plan dimensions, torn receipts, financial tables, web
catalog lookups, graph reading, coffee roast labels, sudoku grid
extraction, sheet music metadata, and school transcripts.

### 1. Read dimensions on a floor plan

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/floor_plan.png")

prediction = run_ocr(image, prompt="What are the dimensions of the living area? Return in format 11'1 X 11'1.")

display_result(
    image=image,
    expected="10'2\" X 16'",
    prediction=prediction,
)

results.append({"name": "Read dimensions on a floor plan", "expected": "10'2\" X 16'", "prediction": prediction})

### 2. Torn receipt reading

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/receipt_subtotal_tear.png")

prediction = run_ocr(image, prompt="What is the subtotal of the receipt? Return only the amount like $1.00 and no other text.")

display_result(
    image=image,
    expected="$11.69",
    prediction=prediction,
)

results.append({"name": "Torn receipt reading", "expected": "$11.69", "prediction": prediction})

### 3. Table understanding

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/table_understanding.png")

prediction = run_ocr(image, prompt="What is the adjusted EBITDA in the nine months ended September 2023 (in millions)?")

display_result(
    image=image,
    expected="2,428",
    prediction=prediction,
)

results.append({"name": "Table understanding", "expected": "2,428", "prediction": prediction})

### 4. Table understanding from a web catalog

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/screenshot_web_catalog.png")

prediction = run_ocr(image, prompt="What is the package quantity of 90183A308? Return only a number. If SKU does not exist, return NULL.")

display_result(
    image=image,
    expected="50",
    prediction=prediction,
)

results.append({"name": "Table understanding from a web catalog", "expected": "50", "prediction": prediction})

### 5. Point a line flattens on a graph

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/epoch_flat.png")

prediction = run_ocr(image, prompt="At what epoch does the line start to flatten out? Return only the epoch number, like 99.")

display_result(
    image=image,
    expected="4",
    prediction=prediction,
)

results.append({"name": "Point a line flattens on a graph", "expected": "4", "prediction": prediction})

### 6. Roast date on a label

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/roast_date.png")

prediction = run_ocr(image, prompt="When was the coffee roasted? Return only the roast date in DD/MM/YYYY, like 01/12/2019.")

display_result(
    image=image,
    expected="08/09/2020",
    prediction=prediction,
)

results.append({"name": "Roast date on a label", "expected": "08/09/2020", "prediction": prediction})

### 7. Sudoku puzzle extraction

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/sudoku.png")

prediction = run_ocr(image, prompt="You are given an image of a Sudoku puzzle. Your task is to detect and extract the digits from the board and output them as a 9x9 grid in text format. Identify the numbers visible on the Sudoku grid. Use a dot (.) to represent empty cells (i.e., those without a digit). Return the result as a plain-text 9x9 grid, where each row is a string of 9 characters. Maintain the correct order from top-left to bottom-right of the board.")

display_result(
    image=image,
    expected="...4.8...\n.6..7..1.\n7.2.9.5.4\n.9.7.4.3.\n..7.5.8..\n.8.9.6.5.\n9.4.1.7.8\n.7..6..4.\n...2.7...",
    prediction=prediction,
)

results.append({"name": "Sudoku puzzle extraction", "expected": "...4.8...\n.6..7..1.\n7.2.9.5.4\n.9.7.4.3.\n..7.5.8..\n.8.9.6.5.\n9.4.1.7.8\n.7..6..4.\n...2.7...", "prediction": prediction})

### 8. Find an author

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/music.png")

prediction = run_ocr(image, prompt="Who transcribed this sheet music, according to the document? Answer as only the name is presented on the document.")

display_result(
    image=image,
    expected="Otto Singer II",
    prediction=prediction,
)

results.append({"name": "Find an author", "expected": "Otto Singer II", "prediction": prediction})

### 9. Total Credits in Fall 2015

In [ ]:
image = download_image("https://raw.githubusercontent.com/roboflow/vision-ai-checkup/main/images/transcript.png")

prediction = run_ocr(image, prompt="Find the total credits in fall 2015/16 in the attached school transcript")

display_result(
    image=image,
    expected="12",
    prediction=prediction,
)

results.append({"name": "Total Credits in Fall 2015", "expected": "12", "prediction": prediction})

## Results Summary

In [ ]:
print(f"{'#':<4} {'Case':<50} {'Match':>5}")
print("-" * 61)

correct = 0
for i, r in enumerate(results, 1):
    expected = str(r['expected']).strip().lower()
    predicted = str(r['prediction']).strip().lower()
    match = expected == predicted
    correct += int(match)
    mark = '✓' if match else '✗'
    print(f"{i:<4} {r['name']:<50} {mark:>5}")

print("-" * 61)
total = len(results)
print(f"\nScore: {correct}/{total} ({100 * correct / total:.1f}%)")